# Data

Code for downloading and pre-processing data.

In [ ]:
import os
import tarfile
from pathlib import Path
from urllib import request

import keras
import librosa
import numpy as np
import pandas as pd

In [ ]:
DATASET_URL = "https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2"
DATASET_ARCHIVE = "ljspeech.tar.bz2"

def setup_dir_structure(project_root: Path):
    """Create a data directory with `raw/` and `features/` subfolders."""
    raw = project_root / "data" / "raw"
    features = project_root / "data" / "features"

    raw.mkdir(parents=True, exist_ok=True)
    features.mkdir(parents=True, exist_ok=True)
    return raw

def download_extract_dataset(output_dir: Path):
    archive_path = output_dir / DATASET_ARCHIVE
    request.urlretrieve(DATASET_URL, archive_path)
    with tarfile.open(archive_path, "r:bz2") as tar:
        tar.extractall(path=output_dir)
    os.remove(archive_path)

raw_dir = setup_dir_structure(Path.cwd().parent)

In [ ]:
download_extract_dataset(raw_dir)

## Model Input Data

### Background

Tacotron is an end-to-end tts model. This means the model takes as input a character sequence, outputting audio. Input to the model is a character sequence, where each character is one-hot encoded and then embedded into a continuous vector. Embeddings are learned as part of training the model. First, simple text normalization should be applied to the input string. Then, each character in the string needs to be one-hot encoded. Finally, sequences of embeddings should be zero-padded to some common length.

The model uses MAE loss for training. Equally weighted losses are computed for the seq2seq decoder (mel-scale spectrogram) and the post-processing net.

### Batching

Batches will be sized at 32 sequences as per the paper. It's helpful for a batch if the name is also extracted from the csv file so that audio data can be extracted later for calculating loss.

In [ ]:
MAX_SEQ_LENGTH = 256
MAX_WAV_LEN_S = 10

def mfcc_from_label(label, wav_root: Path):
    fname = (wav_root / label).with_suffix(".wav")
    y, sr = librosa.load(fname)

    # Pad input
    pad_samples = MAX_WAV_LEN_S * sr - y.shape[0]
    y = np.pad(y, (0, pad_samples))

    frame_size = sr * .05
    hop_size = sr * .0125
    mfcc = librosa.feature.melspectrogram(
        y=y, 
        sr=sr, 
        n_fft=2048, 
        win_length=int(frame_size), 
        hop_length=int(hop_size),
        center=False
    )
    return mfcc

def get_acoustic_features(labels: pd.Series, wav_root: Path):
    return [mfcc_from_label(label, wav_root) for label in labels]

def get_char_features(speech_strings):
    # strings -> ints -> padded -> ohe
    int_seqs = [[ord(ch) for ch in line[:MAX_SEQ_LENGTH]] for line in speech_strings]
    int_seqs_pad = [seq + [0] * (MAX_SEQ_LENGTH - len(seq)) for seq in int_seqs]
    ohe_seqs = keras.utils.to_categorical(int_seqs_pad, num_classes=256)
    return ohe_seqs

def get_features(raw_chunk: pd.DataFrame, wav_root: Path):
    labels = raw_chunk[0]
    text_lines = raw_chunk[2]
    ohe_seqs = get_char_features(text_lines)
    mfccs = get_acoustic_features(labels, wav_root)
    return (ohe_seqs, mfccs)

In [ ]:
SPEECH_LABELS = raw_dir / "LJSpeech-1.1" / "metadata.csv"
WAV_ROOT = raw_dir / "LJSpeech-1.1" / "wavs"

for chunk in pd.read_csv(SPEECH_LABELS, chunksize=32, sep="|", header=None):
    enc_seq, acu_seq = get_features(chunk, WAV_ROOT)
    break